# UC14 — Evaluación Automática de Agentes (QA)

Scoring automático de calidad de llamadas evaluando empatía, cumplimiento de script y normativa.

**Valor de negocio**: Mejora continua de calidad + ahorro del 80% en auditorías manuales.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS QA_AGENTES_DB;
USE SCHEMA QA_AGENTES_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Transcripciones de Llamadas con Agentes

In [ ]:
CREATE OR REPLACE TABLE llamadas_agentes AS
SELECT 'CALL' || LPAD(SEQ4(),5,'0') AS llamada_id,
    'AGT' || LPAD(MOD(SEQ4(),15)+1,3,'0') AS agente_id,
    DATEADD(day,-UNIFORM(1,90,RANDOM()),CURRENT_DATE()) AS fecha,
    UNIFORM(120,600,RANDOM()) AS duracion_seg,
    CASE MOD(SEQ4(),5)
        WHEN 0 THEN 'Agente: Buenos días, le atiende María del departamento de siniestros, ¿en qué puedo ayudarle? Cliente: Hola, tuve un accidente ayer. Agente: Lamento mucho lo ocurrido. ¿Se encuentra usted bien? Vamos a gestionar su siniestro paso a paso. Le informo de que esta llamada será grabada por motivos de calidad. ¿Me permite su número de póliza?'
        WHEN 1 THEN 'Agente: Hola. ¿Qué quiere? Cliente: Necesito abrir un parte. Agente: Dígame los datos. Cliente: Es que estoy muy nervioso, el golpe fue fuerte. Agente: Ya, bueno, pero necesito los datos para continuar. ¿Número de póliza?'
        WHEN 2 THEN 'Agente: Buenas tardes, soy Pedro. ¿En qué puedo ayudarle? Cliente: Quiero cancelar mi seguro, es carísimo. Agente: Entiendo su preocupación por el precio. Antes de proceder, ¿me permitiría revisar su póliza? Quizás podamos encontrar una alternativa que se ajuste mejor a su presupuesto. Le informo de sus derechos como asegurado.'
        WHEN 3 THEN 'Agente: Diga. Cliente: ¿Mi seguro cubre daños por agua? Agente: Pues depende. Mire su póliza. Cliente: Pero eso ya lo he hecho. Agente: Pues entonces no sé qué decirle. Llame a otro departamento.'
        ELSE 'Agente: Buenos días, le atiende Ana. Esta llamada se graba por motivos de calidad y cumplimiento normativo. ¿En qué puedo ayudarle? Cliente: Necesito asistencia en carretera. Agente: Por supuesto. Entiendo que es una situación estresante. Voy a localizar su ubicación y enviar asistencia inmediatamente. ¿Se encuentra en un lugar seguro? Conforme al artículo 12 de su póliza, tiene derecho a vehículo de sustitución.'
    END AS transcripcion
FROM TABLE(GENERATOR(ROWCOUNT=>500));

SELECT agente_id, COUNT(*) AS llamadas, ROUND(AVG(duracion_seg)) AS duracion_media FROM llamadas_agentes GROUP BY agente_id ORDER BY llamadas DESC;

---

## Paso 3: Evaluar Calidad con Cortex AI

Evaluamos 5 criterios: saludo, empatía, cumplimiento normativo, resolución y profesionalismo.

In [ ]:
CREATE OR REPLACE TABLE evaluaciones AS
SELECT llamada_id, agente_id, fecha, duracion_seg,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Eres un evaluador de calidad de un call center de seguros. Evalúa esta transcripción en 5 criterios, puntuando de 1 a 10 cada uno. Responde SOLO con un JSON válido: {"saludo_protocolo": X, "empatia": X, "cumplimiento_normativo": X, "resolucion": X, "profesionalismo": X, "comentario": "máx 20 palabras"}.\n\nTranscripción: ' || transcripcion
    ) AS evaluacion_raw,
    SNOWFLAKE.CORTEX.SENTIMENT(transcripcion) AS sentimiento_score
FROM llamadas_agentes;

SELECT llamada_id, agente_id, sentimiento_score, LEFT(evaluacion_raw,300) AS evaluacion
FROM evaluaciones LIMIT 10;

---

## Paso 4: Parsear Puntuaciones y Calcular Score Global

In [ ]:
CREATE OR REPLACE TABLE scores_agentes AS
SELECT llamada_id, agente_id, fecha, duracion_seg, sentimiento_score,
    TRY_PARSE_JSON(evaluacion_raw) AS ev,
    COALESCE(ev['saludo_protocolo']::INT, 5) AS saludo,
    COALESCE(ev['empatia']::INT, 5) AS empatia,
    COALESCE(ev['cumplimiento_normativo']::INT, 5) AS normativo,
    COALESCE(ev['resolucion']::INT, 5) AS resolucion,
    COALESCE(ev['profesionalismo']::INT, 5) AS profesionalismo,
    ROUND((COALESCE(ev['saludo_protocolo']::INT,5) + COALESCE(ev['empatia']::INT,5) + COALESCE(ev['cumplimiento_normativo']::INT,5) + COALESCE(ev['resolucion']::INT,5) + COALESCE(ev['profesionalismo']::INT,5)) / 5.0, 1) AS score_global,
    COALESCE(ev['comentario']::VARCHAR, '') AS comentario
FROM evaluaciones;

SELECT agente_id, COUNT(*) AS llamadas, ROUND(AVG(score_global),1) AS score_medio,
    ROUND(AVG(saludo),1) AS saludo, ROUND(AVG(empatia),1) AS empatia,
    ROUND(AVG(normativo),1) AS normativo, ROUND(AVG(resolucion),1) AS resolucion
FROM scores_agentes GROUP BY agente_id ORDER BY score_medio DESC;

---

## Paso 5: Identificar Agentes para Coaching

In [ ]:
SELECT agente_id, COUNT(*) AS llamadas, ROUND(AVG(score_global),1) AS score_medio,
    SUM(CASE WHEN score_global < 5 THEN 1 ELSE 0 END) AS llamadas_bajo_5,
    ROUND(MIN(score_global),1) AS peor_score,
    CASE WHEN AVG(score_global) >= 8 THEN 'Excelente'
         WHEN AVG(score_global) >= 6 THEN 'Aceptable'
         WHEN AVG(score_global) >= 4 THEN 'Necesita Mejora'
         ELSE 'Crítico' END AS evaluacion
FROM scores_agentes GROUP BY agente_id ORDER BY score_medio;

---

## Paso 6: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Evaluación Automática de Agentes')
st.markdown('### Quality Assurance con IA — Snowflake Cortex')

df = session.table('scores_agentes').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Llamadas Evaluadas', len(df))
with c2: st.metric('Agentes', df['AGENTE_ID'].nunique())
with c3: st.metric('Score Medio', f"{df['SCORE_GLOBAL'].mean():.1f}/10")
with c4: st.metric('Bajo Umbral (<5)', len(df[df['SCORE_GLOBAL']<5]))

st.markdown('---')
st.subheader('Ranking de Agentes')
ranking = df.groupby('AGENTE_ID').agg({'SCORE_GLOBAL':'mean','SALUDO':'mean','EMPATIA':'mean','NORMATIVO':'mean','RESOLUCION':'mean','LLAMADA_ID':'count'}).round(1).rename(columns={'LLAMADA_ID':'Llamadas'}).sort_values('SCORE_GLOBAL',ascending=False)
st.dataframe(ranking, use_container_width=True)

st.markdown('---')
st.subheader('Detalle por Agente')
agente = st.selectbox('Agente', sorted(df['AGENTE_ID'].unique()))
adf = df[df['AGENTE_ID']==agente].sort_values('SCORE_GLOBAL')
st.dataframe(adf[['LLAMADA_ID','FECHA','SCORE_GLOBAL','SALUDO','EMPATIA','NORMATIVO','RESOLUCION','PROFESIONALISMO','COMENTARIO']], use_container_width=True, height=300)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS QA_AGENTES_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;